# Figure 8

**Effect of gas pressure after uniaxial stretch**

This notebook evaluates the effect of increasing gas pressure on the homogenized permeability after prescribed uniaxial stretches along the $x$-direction, for several initial porosities.

---

**Authors**

Haotian XIAO
Martin GENET

École Polytechnique, Palaiseau, France


## Imports

In [ ]:
from pathlib import Path
import math

import dolfin
import myPythonLibrary as mypy
import dolfin_mech as dmech

## Output directory

In [ ]:
# All results and figures of this notebook are written to the subdirectory Fig8/
# of the notebook directory (demos/).
RESULTS_DIR = Path("Fig8").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR =", RESULTS_DIR)

## Gas-pressure loading function

In [ ]:
def run_gas_pressure_after_stretch_x_test(
    res_folder,
    r0_list=(0.072, 0.41, 0.5),
    Ex_list=(0.05, 0.1, 0.2, 0.3),
    pg_fin=0.2,
    probe_list=("gx", "gy"),
    perform_tests=0,
):
    res_folder = Path(res_folder)
    res_folder.mkdir(parents=True, exist_ok=True)

    test = mypy.Test(
        res_folder=str(res_folder),
        perform_tests=perform_tests,
        stop_at_failure=1,
        clean_after_tests=0,
        tester_numpy_tolerance=1e-2,
    )

    material_parameters = {
        "alpha": 0.16,
        "gamma": 0.5,
        "c1": 0.2,
        "c2": 0.4,
        "kappa": 1e2,
        "eta": 1e-5,
    }

    dim = 2
    bcs = "pbc"
    n_steps = 2

    print("res_folder =", res_folder)
    print("r0_list =", r0_list)
    print("Ex_list =", Ex_list)
    print("pg_fin =", pg_fin)
    print("dim =", dim)
    print("bcs =", bcs)

    def format_float_for_filename(value):
        return f"{value:.6g}"

    def run_one_probe(r0, Ex, probe):
        if probe == "gx":
            gx_ini_lst = [0.0, 0.01]
            gx_fin_lst = [0.01, 0.01]
            gy_ini_lst = [0.0, 0.0]
            gy_fin_lst = [0.0, 0.0]

        elif probe == "gy":
            gx_ini_lst = [0.0, 0.0]
            gx_fin_lst = [0.0, 0.0]
            gy_ini_lst = [0.0, 0.01]
            gy_fin_lst = [0.01, 0.01]

        else:
            raise ValueError("probe must be 'gx' or 'gy'")

        r0_str = format_float_for_filename(r0)
        Ex_str = format_float_for_filename(Ex)

        res_basename = (
            f"pg-after-Ex-r0={r0_str}-Ex={Ex_str}-{probe}"
        )

        result_basename = res_folder / res_basename
        mesh_filebasename = res_folder / f"mesh-r0={r0_str}"

        load_params = {
            "solid": {},
            "liquid": {},
            "air": {},
        }

        for i in range(dim):
            for j in range(dim):
                load_params["solid"][f"sigma_bar_{i}{j}"] = 0.0

        load_params["solid"]["U_bar_00_lst"] = [Ex, Ex]

        load_params["liquid"]["pl_bar_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["pl_bar_fin_lst"] = [0.0] * n_steps

        load_params["liquid"]["grad_p_bar_x_ini_lst"] = gx_ini_lst
        load_params["liquid"]["grad_p_bar_x_fin_lst"] = gx_fin_lst
        load_params["liquid"]["grad_p_bar_y_ini_lst"] = gy_ini_lst
        load_params["liquid"]["grad_p_bar_y_fin_lst"] = gy_fin_lst

        load_params["liquid"]["Theta_in_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_in_fin_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_out_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_out_fin_lst"] = [0.0] * n_steps

        load_params["air"]["pf"] = pg_fin
        load_params["air"]["pf_lst"] = [0.0, pg_fin]

        dmech.run_HollowBox_MicroPoroflow(
            dim=dim,
            mesh_params={
                "dim": dim,
                "xmin": 0.0,
                "ymin": -math.sqrt(3.0) / 2.0,
                "xmax": 1.0,
                "ymax": math.sqrt(3.0) / 2.0,
                "r0": r0,
                "l": 0.03,
                "hole_shape": "hex",
                "add_center_hole": True,
                "mesh_filebasename": str(mesh_filebasename),
            },
            mat_params={
                "skel": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
                "bulk": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
                "pore": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
            },
            flow_params={
                "k_l": dolfin.Constant(
                    (
                        (1e-6, 0.0),
                        (0.0, 1e-6),
                    )
                ),
                "use_kozeny_carman": False,
            },
            porosity_params={
                "type": "constant",
                "val": 0.3,
            },
            bcs=bcs,
            step_params={
                "n_steps": n_steps,
                "Deltat_lst": [1e-2, 1e-1],
                "dt_ini_lst": [1e-3, 1e-3],
                "dt_min_lst": [1e-4, 1e-4],
                "dt_max_lst": [5e-3, 5e-3],
            },
            load_params=load_params,
            res_basename=str(result_basename),
            verbose=0,
        )

        test.test(str(result_basename))

    for r0 in r0_list:
        for Ex in Ex_list:
            print()
            print("========================================")
            print(
                f"Case: r0={r0}, Ex={Ex}, "
                f"stretch-x first, then gas pressure to pg={pg_fin}"
            )
            print("========================================")

            for probe in probe_list:
                print(
                    f"Running r0={r0}, "
                    f"Ex={Ex}, "
                    f"pg_fin={pg_fin}, "
                    f"probe={probe}"
                )

                run_one_probe(
                    r0=r0,
                    Ex=Ex,
                    probe=probe,
                )

    return res_folder

## Parameters

In [ ]:
r0_list = (0.072, 0.41, 0.5)
Ex_list = (0.0, 0.1, 0.2)
pg_fin = 0.5
probe_list = ("gx", "gy")

res_folder = RESULTS_DIR / "Figure8_gas_pressure_after_stretch_x"

## Computing response to gas pressure after uniaxial stretch

In [ ]:
res_folder = run_gas_pressure_after_stretch_x_test(
    res_folder=res_folder,
    r0_list=r0_list,
    Ex_list=Ex_list,
    pg_fin=pg_fin,
    probe_list=probe_list,
)

print("Results saved in:", res_folder)

## Plot Figure 8

In [ ]:
from Plot_Microporoflow import plot_pg_after_stretch_x_summary


def build_figure8(
    res_folder,
    r0_list,
    Ex_list,
    probe_list,
):
    figures_folder = RESULTS_DIR / "figures"
    figures_folder.mkdir(parents=True, exist_ok=True)

    figure_path = figures_folder / "Figure8_pg_after_stretch_x.pdf"

    plot_pg_after_stretch_x_summary(
        res_folder=str(res_folder),
        res_basename_prefix="pg-after-Ex",
        r0_list=r0_list,
        Ex_list=Ex_list,
        probe_list=probe_list,
        stream_r0=0.41,
        stream_Ex=0.2,
        stream_probe="gx",
        n_stream_states=6,
        stream_density=0.8,
        stream_scale=1.0,
        stream_grid_n=500,
        save_name=str(figure_path),
        show_plot=True,
    )

    return figure_path

### Figure generation

In [ ]:
figure_path = build_figure8(
    res_folder=res_folder,
    r0_list=r0_list,
    Ex_list=Ex_list,
    probe_list=probe_list,
)

print("Figure saved in:", figure_path)